In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import nibabel as nib
import os
import tensorflow as tf
from tensorflow.keras.utils import to_categorical # type: ignore # type: ignore
from skimage.transform import resize
from utils import util


In [2]:
print(pd.__version__)
print(np.__version__)
print(tf.__version__)
print(torch.__version__)
print("GPU is", "available" if torch.cuda.is_available() else "NOT AVAILABLE")

3.0.3
1.26.4
2.16.2
2.5.1+cpu
GPU is NOT AVAILABLE


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from itertools import zip_longest
IMG_SIZE = (128, 128)

VOLUME_DIR = util.get_volume_dir()
SEGMENTATION_DIR = util.get_segment_dir()

print(f"path:{os.listdir(SEGMENTATION_DIR)} {os.listdir(VOLUME_DIR)}")


X_train, Y_train = [], []

image_files = sorted([f for f in os.listdir(VOLUME_DIR) if f.endswith(".nii")])
mask_files = sorted([f for f in os.listdir(SEGMENTATION_DIR) if f.endswith(".nii")])

for img_file, mask_file in zip(image_files, mask_files):
    volume_path = os.path.join(VOLUME_DIR, img_file)
    segmentation_path = os.path.join(SEGMENTATION_DIR, mask_file)
    vol, seg = util.preprocess_data(volume_path, segmentation_path)
    X_train.append(vol)
    Y_train.append(seg)

print("Training data shape:", X_train.shape)
print("Segmentation mask shape:", Y_train.shape)

X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)

print("Training data shape:", X_train.shape)
print("Segmentation mask shape:", Y_train.shape)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

# Create dataset
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)

# Create data loader
train_loader = DataLoader(
    train_dataset,
    batch_size=2,   # you can try 2 or 4 depending on GPU memory
    shuffle=True,
    num_workers=0
)

print("Training loader created successfully")

path:['segmentation-0.nii', 'segmentation-1.nii', 'segmentation-11.nii', 'segmentation-12.nii', 'segmentation-13.nii', 'segmentation-14.nii', 'segmentation-16.nii', 'segmentation-17.nii', 'segmentation-18.nii', 'segmentation-19.nii', 'segmentation-2.nii', 'segmentation-20.nii', 'segmentation-21.nii', 'segmentation-22.nii', 'segmentation-23.nii', 'segmentation-3.nii', 'segmentation-4.nii'] ['volume-0.nii', 'volume-1.nii', 'volume-11.nii', 'volume-12.nii', 'volume-13.nii', 'volume-14.nii', 'volume-16.nii', 'volume-17.nii', 'volume-18.nii', 'volume-19.nii', 'volume-2.nii', 'volume-20.nii', 'volume-21.nii', 'volume-22.nii', 'volume-23.nii', 'volume-3.nii', 'volume-4.nii']
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\train\volumes\volume-0.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\train\segmentation\segmentation-0.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model

In [ ]:
# from models.UNetPlusPlus import UNetPlusPlus
# import torch.nn as nn
# import torch.optim as optim

# # Use GPU if available, otherwise CPU
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using device:", device)

# model = UNetPlusPlus(in_channels=1, out_channels=3).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=1e-3)

# num_epochs = 1

# for epoch in range(num_epochs):
#     model.train()
#     epoch_loss = 0.0

#     for batch_idx, (inputs, targets) in enumerate(train_loader):
#         inputs = inputs.to(device)
#         targets = targets.to(device)

#         # Make sure inputs are in (N, C, H, W) format
#         if inputs.ndim == 3:
#             inputs = inputs.unsqueeze(1)
#         elif inputs.ndim == 4 and inputs.shape[-1] == 1:
#             inputs = inputs.permute(0, 3, 1, 2)

#         # Convert masks to class indices for CrossEntropyLoss
#         if targets.ndim == 4 and targets.shape[1] == 3:
#             targets = targets.argmax(dim=1)
#         elif targets.ndim == 4 and targets.shape[-1] == 3:
#             targets = targets.permute(0, 3, 1, 2).argmax(dim=1)
#         elif targets.ndim == 3:
#             targets = targets.long()
#         else:
#             targets = targets.long()

#         outputs = model(inputs)
#         loss = criterion(outputs, targets.long())

#         optimizer.zero_grad(set_to_none=True)
#         loss.backward()
#         optimizer.step()
#         epoch_loss += loss.item()

#         print(f"Batch {batch_idx + 1}/{len(train_loader)} loss: {loss.item():.4f}")

#     avg_loss = epoch_loss / len(train_loader)
#     print(f"Epoch [{epoch + 1}/{num_epochs}] Loss: {avg_loss:.4f}")

Using device: cpu
Batch 1/4380 loss: 1.2119
Batch 2/4380 loss: 1.1437
Batch 3/4380 loss: 1.0621
Batch 4/4380 loss: 1.0313
Batch 5/4380 loss: 1.0483
Batch 6/4380 loss: 0.9362
Batch 7/4380 loss: 0.8671
Batch 8/4380 loss: 0.8323
Batch 9/4380 loss: 0.7859
Batch 10/4380 loss: 0.7953
Batch 11/4380 loss: 0.7467
Batch 12/4380 loss: 0.7212
Batch 13/4380 loss: 0.6844
Batch 14/4380 loss: 0.6816
Batch 15/4380 loss: 0.7222
Batch 16/4380 loss: 0.6643
Batch 17/4380 loss: 0.6235
Batch 18/4380 loss: 0.6007
Batch 19/4380 loss: 0.6422
Batch 20/4380 loss: 0.6311
Batch 21/4380 loss: 0.6591
Batch 22/4380 loss: 0.5665
Batch 23/4380 loss: 0.5297
Batch 24/4380 loss: 0.5675
Batch 25/4380 loss: 0.4947
Batch 26/4380 loss: 0.4959
Batch 27/4380 loss: 0.5138
Batch 28/4380 loss: 0.6100
Batch 29/4380 loss: 0.5299
Batch 30/4380 loss: 0.4451
Batch 31/4380 loss: 0.4689
Batch 32/4380 loss: 0.5137
Batch 33/4380 loss: 0.4120
Batch 34/4380 loss: 0.5370
Batch 35/4380 loss: 0.4206
Batch 36/4380 loss: 0.5203
Batch 37/4380 loss:

In [ ]:
# import os
# import torch

# save_dir = os.path.join(
#     util.get_segment_model_dir(),
#     "checkpoints/UNetPlusPlus"
# )
# os.makedirs(save_dir, exist_ok=True)

# save_path = os.path.join(save_dir, "unetplusplus_liver_segmentation.pth")

# torch.save({
#     "model_state_dict": model.state_dict(),
#     "optimizer_state_dict": optimizer.state_dict(),
#     "num_epochs": num_epochs,
#     "device": device
# }, save_path)

# print(f"Model saved to: {save_path}")

Model saved to: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\notebooks\models\checkpoints/UNetPlusPlus\unetplusplus_liver_segmentation.pth


In [21]:
import os
import torch

checkpoint_path = os.path.join(
    util.get_segment_model_dir(),
    "checkpoints/UNetPlusPlus/unetplusplus_liver_segmentation.pth"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = UNetPlusPlus(in_channels=1, out_channels=3).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()
print(f"Model loaded from: {checkpoint_path}")

Model loaded from: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\notebooks\models\checkpoints/UNetPlusPlus/unetplusplus_liver_segmentation.pth


C:\Users\kreddy\AppData\Local\Temp\ipykernel_17880\3033059682.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=devi

In [22]:
import os
import numpy as np
import nibabel as nib
from tensorflow.keras.utils import to_categorical # type: ignore
import utils  
# ---- Load Test Data ----
test_image_path = util.get_test_volume_dir()
test_mask_path = util.get_test_segment_dir()

# Ensure directories exist
if not os.path.exists(test_image_path) or not os.path.exists(test_mask_path):
    raise FileNotFoundError("Test data folder or segmentation folder not found!")

# Get sorted filenames
image_files = sorted([f for f in os.listdir(test_image_path) if f.endswith(".nii")])
mask_files = sorted([f for f in os.listdir(test_mask_path) if f.endswith(".nii")])

# Check if the number of images and masks match
if len(image_files) != len(mask_files):
    raise ValueError("Mismatch between the number of test images and segmentation masks!")

X_test, Y_test = [], []

# Load and preprocess each test image and mask
for img_file, mask_file in zip(image_files, mask_files):
    volume_path = os.path.join(test_image_path, img_file)  # Correct path joining
    segmentation_path = os.path.join(test_mask_path, mask_file)  # Correct path joining

    vol, seg = util.preprocess_data(volume_path, segmentation_path)
    X_test.append(vol)
    Y_test.append(seg)

# Convert lists to NumPy arrays
X_test = np.concatenate(X_test, axis=0)  # Shape: (num_volumes, num_slices, 128, 128)
Y_test = np.concatenate(Y_test, axis=0)  # Shape: (num_volumes, num_slices, 128, 128, 3)

# Reshape to flatten across all slices
X_test = X_test.reshape(-1, 128, 128, 1)  # Add channel dimension
Y_test = Y_test.reshape(-1, 128, 128, 3)  # Keep segmentation masks in one-hot format

# Print shapes
print("Final Test Data Shape:", X_test.shape)  # (total_slices, 128, 128, 1)
print("Final Test Mask Shape:", Y_test.shape)  # (total_slices, 128, 128, 3)


Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\volumes\volume-24.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\segmentation\segmentation-24.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\volumes\volume-25.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\segmentation\segmentation-25.nii
Final Test Data Shape: (877, 128, 128, 1)
Final Test Mask Shape: (877, 128, 128, 3)


In [23]:
# Convert test data to tensors
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)  # (N, 1, 128, 128)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32).permute(0, 3, 1, 2)  # (N, 3, 128, 128)

# Create a DataLoader with all slices in one batch
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [24]:
def compute_metrics(y_true, y_pred, num_classes=3):
    smooth = 1e-6
    metrics = {}

    for cls in range(num_classes):
        y_true_cls = (y_true == cls)
        y_pred_cls = (y_pred == cls)

        intersection = (y_true_cls & y_pred_cls).sum().item()
        union = (y_true_cls | y_pred_cls).sum().item()
        tp = intersection
        fp = y_pred_cls.sum().item() - tp
        fn = y_true_cls.sum().item() - tp

        precision = tp / (tp + fp + smooth)
        recall = tp / (tp + fn + smooth)
        f1 = 2 * precision * recall / (precision + recall + smooth)
        iou = intersection / (union + smooth)
        dice = 2 * intersection / (y_pred_cls.sum().item() + y_true_cls.sum().item() + smooth)

        metrics[cls] = {
            'IoU': iou,
            'Precision': precision,
            'Recall': recall,
            'F1-score': f1,
            'Dice': dice
        }

    # Dice for class 1 + class 2 (liver + tumor)
    liver = (y_true == 1)
    tumor = (y_true == 2)
    pred_liver = (y_pred == 1)
    pred_tumor = (y_pred == 2)

    combined_true = liver | tumor
    combined_pred = pred_liver | pred_tumor
    intersection = (combined_true & combined_pred).sum().item()
    dice_combined = 2 * intersection / (combined_true.sum().item() + combined_pred.sum().item() + smooth)

    return metrics, dice_combined

In [26]:
model.eval()
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

         # If an extra singleton dim remains, remove it
        if inputs.ndim == 5:
            inputs = inputs.squeeze(-1)

        outputs = model(inputs)
        predictions = torch.argmax(outputs, dim=1)  # (N, 128, 128)
        true_classes = torch.argmax(targets, dim=1)  # (N, 128, 128)

metrics, dice_combined = compute_metrics(true_classes.cpu(), predictions.cpu())

# Print all class-wise metrics
for cls in metrics:
    print(f"Metrics for Class {cls}:")
    print(f"IoU: {metrics[cls]['IoU']:.4f}")
    print(f"Precision: {metrics[cls]['Precision']:.4f}")
    print(f"Recall: {metrics[cls]['Recall']:.4f}")
    print(f"F1-score: {metrics[cls]['F1-score']:.4f}")
    print(f"Dice Coefficient: {metrics[cls]['Dice']:.4f}\n")

print(f"Dice Coefficient for Liver + Tumor combined: {dice_combined:.4f}")

Metrics for Class 0:
IoU: 0.9924
Precision: 0.9950
Recall: 0.9974
F1-score: 0.9962
Dice Coefficient: 0.9962

Metrics for Class 1:
IoU: 0.6950
Precision: 0.8698
Recall: 0.7757
F1-score: 0.8201
Dice Coefficient: 0.8201

Metrics for Class 2:
IoU: 0.0000
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
Dice Coefficient: 0.0000

Dice Coefficient for Liver + Tumor combined: 0.8202
